In [2]:
# -*- coding: utf-8 -*-
"""
@author: Etienne Kras
"""

# generic imports
import sys
import os
import numpy as np
from pathlib import Path
import platform
import geopandas as gpd
import pandas as pd
import time
import geemap
import json
import geojson
import ee

project = "bathymetry"
ee.Initialize(project=project) # use the GEE project-id here

# specific imports
from typing import Any, Dict, List, Optional
from geojson import Feature, FeatureCollection, dump
from shapely.geometry import Polygon, MultiPolygon, shape
from dateutil.relativedelta import *
from google.cloud import storage
from logging import Logger, getLogger
from googleapiclient.discovery import build
from re import sub
from ctypes import ArgumentError
from functools import partial
from dateutil.parser import parse

# custom functionality import without requirement to pip install package
local_path = r"C:\Users\kras\Documents\GitHub\ee-packages-py"  # path to local GitHub clone
sys.path.append(local_path)
from eepackages.applications.bathymetry import Bathymetry
from eepackages import tiler

logger: Logger = getLogger(__name__)

In [3]:
# see scheme at https://github.com/openearth/eo-bathymetry/blob/master/notebooks/rws-bathymetry/acces_api.pdf for a workflow visualization 

sdb_folder = r"11209821-cmems-global-sdb"

# project toggles
if platform.system().startswith("Windows"):
    main_fol = Path(r"p:\\") / sdb_folder
else:
    main_fol = Path("/mnt/p") / sdb_folder  # name of the main local folder 
bucket = "cmems-sdb" # name of the Google Cloud Storage bucket to store files in the cloud
credential_file = main_fol / "00_miscellaneous" / "KEYS" / "bathymetry-543b622ddce7.json" # Cloud Storage credential key
output_fol = r"01_intertidal/02_data/04_calibrated" # name of the overall project
aoi_fol = r"00_miscellaneous/AOI_upscale" #AOIs
mask_fol = r"00_miscellaneous/Feasibility_maps" # masks
project_name = "AOI_WestEurope" # name of the project AoI
draw_AoI = 0 # toggle 1 to draw AoI, 0 to load

# composite image toggles
mode = "intertidal_improved_100m_upscaled" # specify mode, either "intertidal" or "subtidal"
start_date = "2021-01-01" # start date of the composites
stop_date = "2022-01-01" # end date of the composites
compo_int = 12 # composite interval [months]
compo_len = 12 # composite length [months]
scale = 100  # output resolution of the image [m]
crs = "EPSG:4326" # output projection of the image

# tiling options
zoomed_list = [9, 10, 11] # list with zoom levels to be inspected
sel_tile = 1 # idx of chosen tile level in zoomed_list (inspect the map to chose it accordingly)
# note, see https://www.openearth.nl/rws-bathymetry/2019.html; Z9 is optimal size..

# load google credentials, if specified
if not credential_file == "":  
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credential_file)

# load GTSM & gebco data
gtsm_col = ee.FeatureCollection('projects/bathymetry/assets/gtsm_waterlevels')
gebco_image = ee.Image('projects/bathymetry/assets/gebco_2023_hat_lat')

### Retrieving the global intertidal processing area

In [4]:
# draw or load Area of Interest (AoI)

# TODO: take center of AOI input file if present, put in random coordinate and let user find a place and draw a polygon
# TODO: fix horizontal tiling error (DOS) in API (to use multiple tiles) or move to single polygon run if AoI crosses multiple tiles
Map = geemap.Map(center=(54.2, 6.7), zoom=8) # initialize map with base in Hudayriat

if draw_AoI == 1:
    print("Please draw a polygon somewhere in a water body") # identifier
if draw_AoI == 0:
    # open AoI
    print("Loading and visualizing AoI") #identifier
    #AoIee = geemap.geojson_to_ee(os.path.join(main_fol,'AOI',project_name+'.geojson'))

    with open(main_fol / aoi_fol / (project_name + ".geojson"), 'r') as f:
        contents = geojson.loads(f.read())
    AoIee = ee.Geometry.MultiPolygon(contents["features"][0]["geometry"]["coordinates"])

    Map.addLayer(AoIee, {}, "AoI")

Map # show map

Loading and visualizing AoI


Map(center=[54.2, 6.7], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(c…

In [5]:
# (re)construct the AoI

if draw_AoI == 1:
    
    print("Constructing AoI from drawn polygon") # identifier
    
    # get AoI 
    AoIee = ee.FeatureCollection(Map.draw_features) # make featurecollection
    AoI = Polygon(AoIee.getInfo()["features"][0]["geometry"]["coordinates"][0]) # create AoI shapefile

    # export AoI
    features = []
    features.append(Feature(geometry=AoI, properties={"AoI": project_name}))
    AoIjson = FeatureCollection(features)
    with open(main_fol / "AOI" / (project_name + ".geojson"), "w") as f: # geojson
        dump(AoIjson, f)
    gdr = gpd.GeoDataFrame({"properties":{"AoI": project_name}, "geometry": AoI}, crs="EPSG:4326") #shp
    gdr.to_file(main_fol / "AOI" / (project_name+".shp"))
    bounds = ee.Geometry.Polygon([[[a,b] for a, b in zip(*AoI.exterior.coords.xy)]])
    
if draw_AoI == 0:
    print("Reconstructing AoI from loaded file")
    # get AoI
    with open(main_fol / aoi_fol / (project_name+".geojson")) as f:
        AoIjson = geojson.load(f)
    # try: # drawn polygon in this script
    #     AoI = Polygon(AoIjson["features"][0]["geometry"]["coordinates"]) 
    # except: # drawn in QGIS / ArcGIS and written to geojson there (client file)
    #     AoI = Polygon(AoIjson["features"][0]["geometry"]["coordinates"][0])
    # bounds = ee.Geometry.Polygon([[[a,b] for a, b in zip(*AoI.exterior.coords.xy)]])
    bounds = ee.Geometry.MultiPolygon(AoIjson["features"][0]["geometry"]["coordinates"])

    # make list of multipolygons a single one
    # MPL = []
    # for i in AoIjson["features"]:
    #     for j in i["geometry"]["coordinates"]:
    #         MPL.append(j)

    # bounds = ee.Geometry.MultiPolygon(MPL)

Reconstructing AoI from loaded file


### Retrieving the global intertidal processing area (mask)

In [34]:
# make the AoI a gdf dataframe (takes about 5 min), using geojson
gdf_AOI = gpd.GeoDataFrame.from_features(AoIjson)

# retrieve the (buffered) global coastal intertidal mask (geojson)
# allfiles = os.listdir(os.path.join(main_fol, mask_fol, "Buffered\AOI_results"))
# dfs = []
# for i in allfiles:
#     for j in os.listdir(os.path.join(main_fol, mask_fol, "Buffered\AOI_results", i)):
#         if "result.geojson" in j:
#             print(j) 
#             #open geojson
#             with open(os.path.join(main_fol, mask_fol, "Buffered\AOI_results", i, j)) as f:
#                 #data = geojson.load(f)
#                 df = gpd.read_file(f)

#                 # inner join to match the AOI and mask
#                 df_joined = df.sjoin(gdf_AOI, how="inner")

#                 dfs.append(df_joined) # append to list

# # concat the geodfs list
# gdfs = gpd.GeoDataFrame(pd.concat(dfs, ignore_index=True)) 

In [6]:
# retrieve the (buffered) global coastal intertidal mask (geojson), using parquet
allfiles = os.listdir(main_fol / mask_fol / "2024")#r"2023\Buffered\AOI_results")
for i in allfiles:
    if "gebco" in i and '.parquet' in i and "minus2" in i:
        print(i)
        # read parquet file
        df_buf = gpd.read_parquet(main_fol / mask_fol / "2024" / i)#r"2023\Buffered\AOI_results" / i) --> 1.1 million rows (polygons)

        # filter the data (3 = original, 2 = L-W mask, 1 = <HAT >LAT pixel)
        df_org = df_buf[df_buf["pixel_value"] == 3.0] # select only the pixels with value 3.0 (original) --> 415 k rows (polygons)

        # inner join to match the AOI and mask
        # gdfs_buf = df_buf.sjoin(gdf_AOI, how="inner")
        # gdfs_org = df_org.sjoin(gdf_AOI, how="inner")

gebco_2024_latminus2_merge_result.parquet


In [9]:
# make a multipolygon
#df_mp_buf = gpd.GeoDataFrame({'geometry': [MultiPolygon(list(df_buf.geometry))]}) #gdfs_buf
df_mp_org = gpd.GeoDataFrame({'geometry': [MultiPolygon(list(df_org.geometry))]}) #gdfs_org

# make a ee.Geometry.MultiPolygon to have bounds
# data_buf = json.loads(df_mp_buf.to_json())
# feature_collection_buf = geojson.FeatureCollection(data_buf['features'])
# bounds_buf = ee.Geometry.MultiPolygon(feature_collection_buf["features"][0]["geometry"]["coordinates"])

# make a ee.Geometry.MultiPolygon to have bounds
data_org = json.loads(df_mp_org.to_json())
feature_collection_org = geojson.FeatureCollection(data_org['features'])
bounds_org = ee.Geometry.MultiPolygon(feature_collection_org["features"][0]["geometry"]["coordinates"])

# add to map
#Map.addLayer(bounds, {}, "global intertidal feasibility mask")

### Tiling the AOI

In [10]:
# function for tiling the AoI and showing it on the map
def add_tile_bounds(zoom):
    tiled = tiler.get_tiles_for_geometry(bounds, zoom)
    Map.addLayer(tiled.style(width=max(1, 10 - zoom), fillColor= "00000022"), {}, "tiles " + str(zoom))

    return(tiled)

In [11]:
# for LAT-2

#Z9
# tiles9_buf = tiler.get_tiles_for_geometry(bounds_buf, ee.Number(zoomed_list[0]))
# tiles9_org = tiler.get_tiles_for_geometry(bounds_org, ee.Number(zoomed_list[0]))

#Z10
# tiles10_buf = tiler.get_tiles_for_geometry(bounds_buf, ee.Number(zoomed_list[1]))
tiles10_org = tiler.get_tiles_for_geometry(bounds_org, ee.Number(zoomed_list[1]))

# Z11 
# tiles11_buf = tiler.get_tiles_for_geometry(bounds_buf, ee.Number(zoomed_list[2])) 
# tiles11_org = tiler.get_tiles_for_geometry(bounds_org, ee.Number(zoomed_list[2])) 

In [39]:
print("buffered")
# print(tiles9_buf.size().getInfo())
# print(tiles10_buf.size().getInfo())
# print(tiles11_buf.size().getInfo())
print("original")
# print(tiles9_org.size().getInfo())
print(tiles10_org.size().getInfo())
# print(tiles11_org.size().getInfo())

buffered
original
